In [ ]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [ ]:
words = open('names.txt', 'r').read().splitlines()

In [ ]:
words[:10]

In [ ]:
print(len(words))
print(min(len(w) for w in words))
print(max(len(w) for w in words))

In [ ]:
b= {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1
        #print(ch1, ch2)

In [ ]:
sorted(b.items(), key = lambda kv: kv[1], reverse=True)

In [ ]:
# Convert bigram dict to matrix
N = torch.zeros((27,27), dtype=torch.int32)
N

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0

In [ ]:
stoi

In [ ]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [ ]:
itos = {i:s for s,i in stoi.items()}

In [ ]:
plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color="gray")
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color="gray")

In [ ]:
N[0, :] # 1d array of first row; count of each letter starting the words

In [ ]:
# create probabilty vector for this
p = N[0].float()
p = p / p.sum()
p

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos[ix]

In [ ]:
P = (N+1).float() # plus one for model smoothing; otherwise we might have "inf" when dividing at the loss sections
P /= P.sum(1, keepdim=True)
P

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        #print(itos[ix])
        if ix == 0:
            break
    print("".join(out))

In [ ]:
log_likelyhood = 0.0    # at the end we want to minimize the negative log likelyhood
n = 0

# validate
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelyhood += logprob
        n+=1
        #print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'{log_likelyhood=}') # syntax to print the variable with the text as output
nll = -log_likelyhood # negative log likelyhood for loss function
print(f'{nll=}')
print(f'{nll/n=}')  # normalized by length

In [ ]:
# create the training set of bigrams (x,y)
xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
print(xs)
print(ys)

In [ ]:
xenc = F.one_hot(xs, num_classes=27).float() # one hot encoding as we can not feed integers directly into a neural net
xenc

In [ ]:
plt.imshow(xenc)    # the yellow color shows which bit is encoded in each of those examples

In [ ]:
W = torch.randn((27,27), generator=g, requires_grad=True)
xenc @ W

In [ ]:
# forward pass
logits = xenc @ W    # log counts
counts = logits.exp()   # equivalent N
probs = counts / counts.sum(1, keepdim=True) # softmax
print(probs)
loss = -probs[torch.arange(5), ys].log().mean()
print(loss)

In [ ]:
# backward pass
W.grad = None   # efficiently set gradient to zero
loss.backward() # puts gradient info in W.grad

# update
W.data += -0.1 * W.grad

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Create the dataset
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        #print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print("number of examples: ", num)

# Initialize the network
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27,27), generator=g, requires_grad=True)

In [ ]:
# Gradient Descent
for k in range(10):
    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float() # one hot encoding as we can not feed integers directly into a neural net
    logits = xenc @ W    # log counts
    counts = logits.exp()   # equivalent N
    probs = counts / counts.sum(1, keepdim=True) # softmax
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()  # loss computation with regularization
    print(loss.item())

    # backward pass
    W.grad = None   # efficiently set gradient to zero
    loss.backward() # puts gradient info in W.grad

    # update
    W.data += -50 * W.grad